In [36]:
import pandas as pd
df=pd.read_csv("/content/02) IMDB Dataset.csv")

In [37]:
print(df.shape)
df.head()
df.isnull().sum()
df.drop_duplicates(inplace=True)

(50000, 2)


In [38]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [39]:
df["review"]=df["review"].str.lower()

In [40]:
import re
def remove_urls(text):
  text=re.sub(r"http\S+","",text)
  return text

df["review"]=df["review"].apply(remove_urls)

In [41]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

In [42]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

In [43]:
# remove stopwords

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopword(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")
  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text;

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


NLTK (Natural Language Toolkit) is a library specifically for processing human language.

word_tokenize(text) splits a sentence into individual words (tokens). "I love movies" → ["I", "love", "movies"]
Stopwords are common words that carry little meaning: "the", "is", "a", "in", "it". Removing them leaves the meaningful words.

In [44]:
# Stemming

from nltk.stem import PorterStemmer

def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]
  tokens=word_tokenize(text)
  for word in tokens:
    stemmed_words.append(ps.stem(word))
  return " ".join(stemmed_words)


Stemming cuts words to their root form: "running" → "run", "loved" → "love", "happily" → "happi". This reduces the vocabulary and groups similar words together. " ".join(list) glues a list back into a string with spaces.

In [45]:
# Label Encoding

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]


The sentiment column has text values: "positive" or "negative". Neural networks need numbers, not words. LabelEncoder converts: "negative" → 0, "positive" → 1. .fit_transform() = learn the mapping AND apply it in one step.


In [46]:
# TF- IDF vectorisation
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=5000)
X=tfidf.fit_transform(df["review"])

A model can't understand raw words — it needs numbers. TF-IDF converts each review into a list of 5000 numbers, where each number represents how important a particular word is in that review vs. all other reviews.

TF = Term Frequency (how often does this word appear in this review?)
IDF = Inverse Document Frequency (how rare is this word across all reviews?)
max_features=5000 means we only keep the 5000 most important words.

After this, each review is a vector (array) of 5000 numbers.

In [47]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

We split data into two groups:

Training set — the model learns from this (80%)
Test set — we use this to check how well it learned (20%), but the model never sees it during training

test_size=0.2 = 20% goes to testing. random_state=42 = fixed random seed, so the split is the same every time you run it (reproducibility).

In [48]:
# DataLoader

In [49]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()   # sparse matrix → regular numpy array
X_test = X_test.toarray()
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

PyTorch is the deep learning library used to build and train neural networks.

.toarray() — TF-IDF gives a "sparse matrix" (stores only non-zero values to save memory). We convert it to a regular full array.
torch.from_numpy(...) — converts a numpy array into a PyTorch Tensor (PyTorch's version of a numpy array).
.float() — ensures numbers are 32-bit floats (the standard for neural networks).
TensorDataset — pairs your inputs (X) and labels (y) together.
DataLoader — feeds the data in batches of 64 during training. shuffle=True randomizes order each epoch so the model doesn't memorize sequence.

In [50]:
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [51]:

input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

for epoch in range(10):
    model.train()
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

In [52]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.68780881314913
